
# 02_SOLVER_v1 — Solver ALOC_REC por Rateio Proporcional com Rastreabilidade

## Registro das regras desta versão

Esta versão **v1** foi criada para diferenciar do módulo anterior baseado em otimização linear (`linprog`).

### Escopo

1. Processa **somente o primeiro `MES_REF`** da base.
2. Balanceia capacidade **somente por `ALOC_REC`**.
3. `COD_FER_UNID` ainda **não é etapa de solver** nesta versão; ele permanece como informação de roteiro/auditoria e participa da capacidade via `HOR_FER`.
4. A entrada é uma única base: `bd_LTP_NEC_SOLVER.xlsx`.
5. A saída é a planilha `bd_SOLVER_ALOC_REC_V1.xlsx`.

### Regras de demanda e roteiro

6. `NEC_PCS` é a demanda do item no mês.
7. `NEC_PCS` pertence ao nível `MES_REF` + `ID_PROD_UNID_FAT`.
8. Roteiros e recursos são formas possíveis de atender o `NEC_PCS`.
9. A ordem de tentativa das alternativas é menor `PRIOR_MATPAR` e depois menor `PRIOR_ROT`.

### Regra oficial de capacidade

10. `HOR_CAP = menor valor entre HOR_REC e HOR_FER`.
11. Não existe fallback nesta versão.
12. Se `HOR_REC = 0` ou `HOR_FER = 0`, então `HOR_CAP = 0`.
13. Alternativas com `HOR_CAP <= 0` não entram no plano de produção, mas ficam registradas em auditoria/diagnóstico.

### Regra de balanceamento

14. A cada rodada de prioridade (`PRIOR_MATPAR`, `PRIOR_ROT`), o motor pega os itens com saldo pendente.
15. Os itens são agrupados por `ALOC_REC`.
16. Se a demanda em horas do `ALOC_REC` couber na capacidade restante, atende integralmente a solicitação daquela rodada.
17. Se a demanda em horas ultrapassar a capacidade restante, aplica rateio proporcional:

```text
FATOR_RATEIO_ALOC_REC = CAPACIDADE_RESTANTE_ALOC_REC / HR_DEMANDA_TOTAL_ALOC_REC
```

18. O mesmo fator é aplicado a todos os itens que disputam o mesmo `ALOC_REC` naquela rodada.
19. O saldo não atendido segue para as próximas alternativas.
20. Não existe faixa, meta ou mínimo artificial nesta versão.

### Rastreabilidade inteligente

21. A guia `06_RASTRO_RATEIO` mostra o filme da decisão: item, rodada, recurso, demanda antes, capacidade antes, fator, produção e saldo depois.
22. A guia `07_DIAGNOSTICO_ITEM` resume por item: alternativas, rodadas tentadas, recursos usados, percentual atendido e motivo do saldo final.
23. A guia `08_DIAGNOSTICO_ALTERNATIVAS` mostra todas as alternativas do item e se cada uma estava válida ou bloqueada por capacidade/produtividade.

### Guias geradas

- `01_RESUMO_MAQUINAS`
- `02_PLANO_PRODUCAO`
- `03_NAO_ATEND_ITEM`
- `04_AUDITORIA_CAPACIDADE`
- `05_ALTERNATIVAS_ITEM`
- `06_RASTRO_RATEIO`
- `07_DIAGNOSTICO_ITEM`
- `08_DIAGNOSTICO_ALTERNATIVAS`
- `99_VALIDACAO`


In [ ]:

# ============================================================
# 00. CONFIGURAÇÃO
# ============================================================

print("02_SOLVER_v1_RATEIO_PROPORCIONAL_ALOC_REC_COM_RASTRO")

from pathlib import Path
import numpy as np
import pandas as pd
from openpyxl import load_workbook
try:
    from IPython.display import display
except Exception:
    display = print

# Caminho padrão do projeto no Windows
OUTPUT_DIR_WINDOWS = Path(r"C:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT")
INPUT_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / "bd_SOLVER_ALOC_REC_V1.xlsx"

# Fallback para execução em ambiente local/sandbox
INPUT_FILE_FALLBACK = Path("/mnt/data/bd_LTP_NEC_SOLVER.xlsx")
OUTPUT_SOLVER_FILE_FALLBACK = Path("/mnt/data/bd_SOLVER_ALOC_REC_V1.xlsx")

if INPUT_FILE_WINDOWS.exists():
    INPUT_FILE = INPUT_FILE_WINDOWS
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_WINDOWS
else:
    INPUT_FILE = INPUT_FILE_FALLBACK
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_FALLBACK

print("INPUT_FILE:", INPUT_FILE)
print("OUTPUT_SOLVER_FILE:", OUTPUT_SOLVER_FILE)

# Flags mantidas do processo anterior
LOTE_MIN_FLAG = True
MULTIPLO_EMB_FLAG = True

TOL = 1e-7


In [ ]:

# ============================================================
# 01. FUNÇÕES AUXILIARES
# ============================================================

def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def ensure_columns(df: pd.DataFrame, cols, default=0):
    for c in cols:
        if c not in df.columns:
            df[c] = default
    return df


def calcular_nec_pcs_se_necessario(df: pd.DataFrame) -> pd.DataFrame:
    '''Calcula NEC_PCS somente se a coluna não existir na base.

    Regra herdada do processo anterior:
    NEC_PCS = demanda/carteira/previsão/estoque segurança
              - estoques/transferências/origens/triangulações
              + previsão próximo mês quando MESMA_REG = NAO
              + LTP_COMP_NEC_PCS
              respeitando LIMIT_PCS, lote mínimo e múltiplo de embalagem.
    '''
    df = df.copy()

    if "NEC_PCS" in df.columns:
        df["NEC_PCS"] = to_num(df["NEC_PCS"])
        print("NEC_PCS lido da base de entrada.")
        return df

    required = [
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_COMP_NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)

    for c in required:
        df[c] = to_num(df[c])

    base_comum = (
        df["LTP_CART_ARR_MES_ANT"]
        + df["LTP_CART_MES_ATUAL"]
        + df["LTP_SALDO_PREV_PCS"]
        + df["LTP_EST_SEG_PCS"]
        - df["LTP_EST_INI_PCS"]
        - df["LTP_EST_TRANS_PCS"]
        - df["ORI_TOT_PCS"]
        - df["TRIANG_TOT_PCS"]
    )

    mesma_reg_nao = df.get("MESMA_REG", "SIM").astype(str).str.upper().eq("NAO")

    nec = np.where(
        mesma_reg_nao,
        base_comum + df["LTP_SALDO_PREV_PROX_MES_PCS"],
        base_comum
    )

    nec = pd.Series(nec, index=df.index).clip(lower=0)
    nec = nec + df["LTP_COMP_NEC_PCS"]
    nec = np.maximum(nec, df["LIMIT_PCS"])

    if LOTE_MIN_FLAG:
        mask_lote = (nec > 0) & (df["LOTE_MIN"] > 0)
        nec = np.where(mask_lote, np.maximum(nec, df["LOTE_MIN"]), nec)

    if MULTIPLO_EMB_FLAG:
        tipo = df.get("TIPO_PROD", "").astype(str).str.upper()
        mask_emb = (nec > 0) & (df["QTD_EMB"] > 0) & (tipo.isin(["PA", "MR"]))
        nec = np.where(mask_emb, np.ceil(nec / df["QTD_EMB"]) * df["QTD_EMB"], nec)

    df["NEC_PCS"] = pd.Series(nec, index=df.index).fillna(0)
    print("NEC_PCS calculado internamente porque a base não trouxe a coluna.")
    return df


def preparar_base(df: pd.DataFrame):
    df = df.copy()

    # Numéricos principais
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER", "NEC_PCS"]:
        if c in df.columns:
            df[c] = to_num(df[c])

    # Capacidade oficial da alternativa, sem fallback
    df["HOR_CAP"] = df[["HOR_REC", "HOR_FER"]].min(axis=1)

    # Primeiro mês
    mes_ref = df["MES_REF"].min()
    bd_mes = df[df["MES_REF"].eq(mes_ref)].copy()

    print("MES_REF processado:", mes_ref)
    print("Linhas bd_mes:", len(bd_mes))

    return bd_mes


In [ ]:

# ============================================================
# 02. LEITURA DA BASE
# ============================================================

bd = pd.read_excel(INPUT_FILE)
print("Base lida:", bd.shape)
print("Colunas:", len(bd.columns))

bd = calcular_nec_pcs_se_necessario(bd)
bd_mes = preparar_base(bd)


In [ ]:

# ============================================================
# 03. MONTAGEM DA DEMANDA E DAS ALTERNATIVAS
# ============================================================

# Demanda única por item
meta_cols = [
    "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD",
    "UNID_PROD", "UNID_FAT", "TIPO_PROD"
]

bd_demanda_solver = (
    bd_mes
    .sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
    .agg({
        **{c: "first" for c in meta_cols if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in bd_mes.columns},
        "NEC_PCS": "max"
    })
)

bd_demanda_solver = bd_demanda_solver[bd_demanda_solver["NEC_PCS"] > TOL].copy()

# Alternativas de produção/auditoria
alt_cols = [
    "MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT",
    "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP",
    "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
    "ID_RECURSO", "ID_FERRAMENTA"
]

bd_alternativas_item = bd_mes[[c for c in alt_cols if c in bd_mes.columns]].copy()
bd_alternativas_item = bd_alternativas_item[
    bd_alternativas_item["ID_PROD_UNID_FAT"].isin(bd_demanda_solver["ID_PROD_UNID_FAT"])
].copy()

bd_alternativas_item = bd_alternativas_item.drop_duplicates(
    [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in bd_alternativas_item.columns]
).reset_index(drop=True)

# Alternativas válidas para produção
bd_alternativas_validas = bd_alternativas_item[
    bd_alternativas_item["ALOC_REC"].notna()
    & (bd_alternativas_item["PCS_HORA"] > 0)
    & (bd_alternativas_item["HOR_CAP"] > 0)
].copy()

print("Itens com demanda:", len(bd_demanda_solver))
print("Alternativas auditadas:", len(bd_alternativas_item))
print("Alternativas válidas para produção:", len(bd_alternativas_validas))
print("Alternativas com HOR_CAP zero:", (bd_alternativas_item["HOR_CAP"] <= 0).sum())


In [ ]:

# ============================================================
# 04. SOLVER V1 - RATEIO PROPORCIONAL POR ALOC_REC COM RASTRO
# ============================================================

def executar_rateio_proporcional_aloc_rec(
    bd_demanda: pd.DataFrame,
    bd_alternativas_validas: pd.DataFrame
):
    '''Executa o balanceamento por rateio proporcional com rastro completo.'''
    demanda = bd_demanda.copy()
    alternativas = bd_alternativas_validas.copy()

    cap_aloc = (
        alternativas
        .groupby("ALOC_REC", as_index=False)
        .agg(HOR_REC=("HOR_REC", "max"), HOR_FER=("HOR_FER", "max"))
    )
    cap_aloc["HOR_CAP"] = cap_aloc[["HOR_REC", "HOR_FER"]].min(axis=1)

    capacidade_restante = cap_aloc.set_index("ALOC_REC")["HOR_CAP"].to_dict()
    pendente = demanda.set_index("ID_PROD_UNID_FAT")["NEC_PCS"].to_dict()

    producao = []
    rastro = []

    niveis_prioridade = list(
        alternativas[["PRIOR_MATPAR", "PRIOR_ROT"]]
        .drop_duplicates()
        .sort_values(["PRIOR_MATPAR", "PRIOR_ROT"])
        .itertuples(index=False, name=None)
    )

    for rodada, (prior_matpar, prior_rot) in enumerate(niveis_prioridade, start=1):
        nivel = alternativas[
            alternativas["PRIOR_MATPAR"].eq(prior_matpar)
            & alternativas["PRIOR_ROT"].eq(prior_rot)
        ].copy()

        nivel["RODADA"] = rodada
        nivel["QTD_PENDENTE_ANTES"] = nivel["ID_PROD_UNID_FAT"].map(pendente).fillna(0)
        nivel = nivel[nivel["QTD_PENDENTE_ANTES"] > TOL].copy()

        if nivel.empty:
            continue

        qtd_alt_mesmo_nivel = nivel.groupby("ID_PROD_UNID_FAT")["ALOC_REC"].transform("count")
        nivel["QTD_ALTERNATIVAS_MESMO_NIVEL"] = qtd_alt_mesmo_nivel
        nivel["QTD_SOLICITADA_BRUTA"] = nivel["QTD_PENDENTE_ANTES"] / qtd_alt_mesmo_nivel
        nivel["HR_DEMANDA_BRUTA"] = nivel["QTD_SOLICITADA_BRUTA"] / nivel["PCS_HORA"]

        # Limite próprio da alternativa/rota, separado do rateio do ALOC_REC.
        nivel["HR_SOLICITADA"] = np.minimum(nivel["HR_DEMANDA_BRUTA"], nivel["HOR_CAP"])
        nivel["QTD_SOLICITADA"] = nivel["HR_SOLICITADA"] * nivel["PCS_HORA"]
        nivel["FATOR_LIMITE_ROTA"] = np.where(
            nivel["HR_DEMANDA_BRUTA"] > TOL,
            nivel["HR_SOLICITADA"] / nivel["HR_DEMANDA_BRUTA"],
            0
        )

        for aloc_rec, grupo in nivel.groupby("ALOC_REC", sort=True):
            cap_antes = float(capacidade_restante.get(aloc_rec, 0))
            hr_solicitada_total = float(grupo["HR_SOLICITADA"].sum())
            hr_demanda_bruta_total = float(grupo["HR_DEMANDA_BRUTA"].sum())

            if cap_antes <= TOL:
                fator_rateio = 0.0
                status_grupo = "NAO_ALOCADO_CAP_RESTANTE_ZERO"
            elif hr_solicitada_total <= TOL:
                fator_rateio = 0.0
                status_grupo = "NAO_ALOCADO_HR_SOLICITADA_ZERO"
            else:
                fator_rateio = min(1.0, cap_antes / hr_solicitada_total)
                if fator_rateio < 1 - TOL:
                    status_grupo = "ALOCADO_RATEADO_POR_ALOC_REC"
                elif (grupo["FATOR_LIMITE_ROTA"] < 1 - TOL).any():
                    status_grupo = "ALOCADO_PARCIAL_POR_LIMITE_ROTA"
                else:
                    status_grupo = "ALOCADO_INTEGRAL_RODADA"

            aloc = grupo.copy()
            aloc["CAP_RESTANTE_ALOC_REC_ANTES"] = cap_antes
            aloc["HR_DEMANDA_BRUTA_TOTAL_ALOC_REC"] = hr_demanda_bruta_total
            aloc["HR_DEMANDA_TOTAL_ALOC_REC"] = hr_solicitada_total
            aloc["FATOR_RATEIO_ALOC_REC"] = fator_rateio
            aloc["QTD_PRODUZIR_SOLVER"] = aloc["QTD_SOLICITADA"] * fator_rateio
            aloc["HR_PRODUZIR_SOLVER"] = aloc["HR_SOLICITADA"] * fator_rateio

            hr_produzida_grupo = float(aloc["HR_PRODUZIR_SOLVER"].sum())
            cap_depois = max(0.0, cap_antes - hr_produzida_grupo)
            aloc["CAP_RESTANTE_ALOC_REC_DEPOIS"] = cap_depois
            aloc["STATUS_DECISAO"] = status_grupo

            cond_limite_rota = aloc["FATOR_LIMITE_ROTA"] < 1 - TOL
            cond_rateio = fator_rateio < 1 - TOL
            aloc["MOTIVO_CORTE"] = np.select(
                [cond_limite_rota & cond_rateio, cond_limite_rota, cond_rateio],
                ["LIMITE_ROTA_E_RATEIO_ALOC_REC", "LIMITE_HOR_CAP_DA_ROTA", "RATEIO_ALOC_REC"],
                default="SEM_CORTE_NA_RODADA"
            )

            produzido_item = aloc.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].sum()
            for item, qtd in produzido_item.items():
                pendente[item] = max(0.0, pendente.get(item, 0) - float(qtd))

            aloc["QTD_PENDENTE_DEPOIS"] = aloc["ID_PROD_UNID_FAT"].map(pendente).fillna(0)
            rastro.append(aloc.copy())

            aloc_prod = aloc[aloc["QTD_PRODUZIR_SOLVER"] > TOL].copy()
            if not aloc_prod.empty:
                producao.append(aloc_prod)

            capacidade_restante[aloc_rec] = cap_depois

    if producao:
        bd_plano = pd.concat(producao, ignore_index=True)
    else:
        bd_plano = pd.DataFrame(columns=list(alternativas.columns))

    if rastro:
        bd_rastro = pd.concat(rastro, ignore_index=True)
    else:
        bd_rastro = pd.DataFrame(columns=list(alternativas.columns))

    return bd_plano, cap_aloc, bd_rastro

bd_plano_producao, bd_cap_aloc_rec, bd_rastro_rateio = executar_rateio_proporcional_aloc_rec(
    bd_demanda_solver,
    bd_alternativas_validas
)

print("Linhas plano produção:", len(bd_plano_producao))
print("ALOC_REC com capacidade:", len(bd_cap_aloc_rec))
print("Linhas rastro rateio:", len(bd_rastro_rateio))


In [ ]:

# ============================================================
# 05. SAÍDAS DE NEGÓCIO E RASTREABILIDADE
# ============================================================

cols_plano = [
    "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
    "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID",
    "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP",
    "QTD_SOLICITADA", "HR_SOLICITADA", "FATOR_RATEIO_ALOC_REC",
    "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "MOTIVO_CORTE", "STATUS_DECISAO"
]
plano_producao = bd_plano_producao[[c for c in cols_plano if c in bd_plano_producao.columns]].copy()

atendido_item = (
    plano_producao.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(QTD_ATENDIDA_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"))
    if not plano_producao.empty else pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "QTD_ATENDIDA_SOLVER"])
)
nao_atend_item = bd_demanda_solver.merge(atendido_item, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
nao_atend_item["QTD_ATENDIDA_SOLVER"] = nao_atend_item["QTD_ATENDIDA_SOLVER"].fillna(0)
nao_atend_item["QTD_NAO_ATEND_SOLVER"] = (nao_atend_item["NEC_PCS"] - nao_atend_item["QTD_ATENDIDA_SOLVER"]).clip(lower=0)
nao_atend_item["PERC_ATENDIDO_SOLVER"] = np.where(nao_atend_item["NEC_PCS"] > 0, nao_atend_item["QTD_ATENDIDA_SOLVER"] / nao_atend_item["NEC_PCS"], 0)
cols_nao_atend = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS", "QTD_ATENDIDA_SOLVER", "QTD_NAO_ATEND_SOLVER", "PERC_ATENDIDO_SOLVER"]
nao_atend_item = nao_atend_item[[c for c in cols_nao_atend if c in nao_atend_item.columns]].copy()

prod_maquina = (
    plano_producao.groupby("ALOC_REC", as_index=False).agg(
        HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"),
        QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"),
        QTD_ITENS=("ID_PROD_UNID_FAT", "nunique")
    )
    if not plano_producao.empty else pd.DataFrame(columns=["ALOC_REC", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"])
)
resumo_maquinas = bd_cap_aloc_rec.merge(prod_maquina, on="ALOC_REC", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0})
resumo_maquinas["OCUPACAO_SOLVER_PCT"] = np.where(resumo_maquinas["HOR_CAP"] > 0, resumo_maquinas["HR_PRODUZIR_SOLVER"] / resumo_maquinas["HOR_CAP"], 0)
resumo_maquinas["HR_OCIOSA_SOLVER"] = resumo_maquinas["HOR_CAP"] - resumo_maquinas["HR_PRODUZIR_SOLVER"]
resumo_maquinas["ESTOURO_HR_SOLVER"] = np.where(resumo_maquinas["HR_PRODUZIR_SOLVER"] > resumo_maquinas["HOR_CAP"] + TOL, resumo_maquinas["HR_PRODUZIR_SOLVER"] - resumo_maquinas["HOR_CAP"], 0)
resumo_maquinas = resumo_maquinas[["ALOC_REC", "HOR_REC", "HOR_FER", "HOR_CAP", "HR_PRODUZIR_SOLVER", "OCUPACAO_SOLVER_PCT", "HR_OCIOSA_SOLVER", "ESTOURO_HR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"]].sort_values(["OCUPACAO_SOLVER_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

cols_auditoria = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID", "HOR_REC", "HOR_FER", "HOR_CAP"]
auditoria_capacidade = bd_alternativas_item[[c for c in cols_auditoria if c in bd_alternativas_item.columns]].copy()

cols_alternativas = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP"]
alternativas_item = bd_alternativas_item[[c for c in cols_alternativas if c in bd_alternativas_item.columns]].copy()

cols_rastro = [
    "RODADA", "MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD",
    "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP",
    "QTD_PENDENTE_ANTES", "QTD_ALTERNATIVAS_MESMO_NIVEL", "QTD_SOLICITADA_BRUTA", "HR_DEMANDA_BRUTA", "FATOR_LIMITE_ROTA",
    "QTD_SOLICITADA", "HR_SOLICITADA", "HR_DEMANDA_BRUTA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_ALOC_REC",
    "CAP_RESTANTE_ALOC_REC_ANTES", "FATOR_RATEIO_ALOC_REC", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER",
    "CAP_RESTANTE_ALOC_REC_DEPOIS", "QTD_PENDENTE_DEPOIS", "STATUS_DECISAO", "MOTIVO_CORTE"
]
rastro_rateio = bd_rastro_rateio[[c for c in cols_rastro if c in bd_rastro_rateio.columns]].copy()

alternativas_diag = bd_alternativas_item.copy()
alternativas_diag["ALTERNATIVA_VALIDA_PRODUCAO"] = alternativas_diag["ALOC_REC"].notna() & (alternativas_diag["PCS_HORA"] > 0) & (alternativas_diag["HOR_CAP"] > 0)
alternativas_diag["MOTIVO_INVALIDA"] = np.select(
    [alternativas_diag["ALOC_REC"].isna(), alternativas_diag["PCS_HORA"] <= 0, alternativas_diag["HOR_CAP"] <= 0],
    ["SEM_ALOC_REC", "PCS_HORA_ZERO_OU_INVALIDO", "HOR_CAP_ZERO_OU_NEGATIVO"],
    default="VALIDA_PRODUCAO"
)
prod_alt = (
    plano_producao.groupby(["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"], as_index=False).agg(QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"))
    if not plano_producao.empty else pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER"])
)
alternativas_diag = alternativas_diag.merge(prod_alt, on=["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"], how="left")
alternativas_diag["QTD_PRODUZIR_SOLVER"] = alternativas_diag["QTD_PRODUZIR_SOLVER"].fillna(0)
alternativas_diag["HR_PRODUZIR_SOLVER"] = alternativas_diag["HR_PRODUZIR_SOLVER"].fillna(0)
alternativas_diag["USADA_NO_PLANO"] = alternativas_diag["QTD_PRODUZIR_SOLVER"] > TOL
cols_alt_diag = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "ALTERNATIVA_VALIDA_PRODUCAO", "MOTIVO_INVALIDA", "USADA_NO_PLANO", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER"]
diagnostico_alternativas = alternativas_diag[[c for c in cols_alt_diag if c in alternativas_diag.columns]].copy()

alt_counts = diagnostico_alternativas.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(QTD_ALTERNATIVAS_AUDITADAS=("ALOC_REC", "count"), QTD_ALTERNATIVAS_VALIDAS=("ALTERNATIVA_VALIDA_PRODUCAO", "sum"), QTD_ALTERNATIVAS_USADAS=("USADA_NO_PLANO", "sum"))
rastro_counts = (
    rastro_rateio.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(QTD_RODADAS_TENTADAS=("RODADA", "nunique"), QTD_ALOC_REC_TENTADOS=("ALOC_REC", "nunique"), QTD_ALOC_REC_COM_PRODUCAO=("QTD_PRODUZIR_SOLVER", lambda s: (s > TOL).sum()), PRIMEIRA_RODADA_TENTADA=("RODADA", "min"), ULTIMA_RODADA_TENTADA=("RODADA", "max"), ULTIMO_STATUS_DECISAO=("STATUS_DECISAO", "last"), ULTIMO_MOTIVO_CORTE=("MOTIVO_CORTE", "last"))
    if not rastro_rateio.empty else pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT"])
)
diagnostico_item = nao_atend_item.merge(alt_counts, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left").merge(rastro_counts, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
for c in ["QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS", "QTD_ALTERNATIVAS_USADAS", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_ALOC_REC_COM_PRODUCAO"]:
    if c in diagnostico_item.columns:
        diagnostico_item[c] = diagnostico_item[c].fillna(0)
diagnostico_item["PERC_NAO_ATENDIDO_SOLVER"] = 1 - diagnostico_item["PERC_ATENDIDO_SOLVER"]

def classificar_motivo_saldo(row):
    if row.get("QTD_NAO_ATEND_SOLVER", 0) <= TOL:
        return "ATENDIDO_TOTAL"
    if row.get("QTD_ALTERNATIVAS_AUDITADAS", 0) <= 0:
        return "SEM_ALTERNATIVA_NA_BASE"
    if row.get("QTD_ALTERNATIVAS_VALIDAS", 0) <= 0:
        return "SEM_ALTERNATIVA_VALIDA_PRODUCAO"
    if row.get("QTD_ATENDIDA_SOLVER", 0) <= TOL:
        return "NAO_ATENDIDO_APOS_ALTERNATIVAS_VALIDAS"
    return "SALDO_APOS_RATEIO_E_LIMITES_DE_CAPACIDADE"

def explicacao_curta(row):
    if row["MOTIVO_SALDO_FINAL"] == "ATENDIDO_TOTAL":
        return "Item atendido integralmente."
    if row["MOTIVO_SALDO_FINAL"] == "SEM_ALTERNATIVA_VALIDA_PRODUCAO":
        return "Item tem demanda, mas não possui alternativa produtiva válida com PCS_HORA > 0 e HOR_CAP > 0."
    if row["QTD_ATENDIDA_SOLVER"] <= TOL:
        return "Item passou pelas alternativas válidas, mas não recebeu produção; verificar 06_RASTRO_RATEIO e capacidade restante."
    return "Item recebeu produção parcial; saldo final permaneceu após rateio/limites de capacidade nas alternativas processadas."

diagnostico_item["MOTIVO_SALDO_FINAL"] = diagnostico_item.apply(classificar_motivo_saldo, axis=1)
diagnostico_item["EXPLICACAO_CURTA"] = diagnostico_item.apply(explicacao_curta, axis=1)
cols_diag_item = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS", "QTD_ATENDIDA_SOLVER", "QTD_NAO_ATEND_SOLVER", "PERC_ATENDIDO_SOLVER", "PERC_NAO_ATENDIDO_SOLVER", "QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS", "QTD_ALTERNATIVAS_USADAS", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_ALOC_REC_COM_PRODUCAO", "PRIMEIRA_RODADA_TENTADA", "ULTIMA_RODADA_TENTADA", "ULTIMO_STATUS_DECISAO", "ULTIMO_MOTIVO_CORTE", "MOTIVO_SALDO_FINAL", "EXPLICACAO_CURTA"]
diagnostico_item = diagnostico_item[[c for c in cols_diag_item if c in diagnostico_item.columns]].copy()

print("01_RESUMO_MAQUINAS:", resumo_maquinas.shape)
print("02_PLANO_PRODUCAO:", plano_producao.shape)
print("03_NAO_ATEND_ITEM:", nao_atend_item.shape)
print("04_AUDITORIA_CAPACIDADE:", auditoria_capacidade.shape)
print("05_ALTERNATIVAS_ITEM:", alternativas_item.shape)
print("06_RASTRO_RATEIO:", rastro_rateio.shape)
print("07_DIAGNOSTICO_ITEM:", diagnostico_item.shape)
print("08_DIAGNOSTICO_ALTERNATIVAS:", diagnostico_alternativas.shape)


In [ ]:

# ============================================================
# 06. VALIDAÇÕES
# ============================================================

nec_total = nao_atend_item["NEC_PCS"].sum()
atend_total = nao_atend_item["QTD_ATENDIDA_SOLVER"].sum()
nao_atend_total = nao_atend_item["QTD_NAO_ATEND_SOLVER"].sum()
dif_fechamento = nec_total - atend_total - nao_atend_total

validacao = pd.DataFrame({
    "METRICA": [
        "NEC_PCS_TOTAL", "QTD_ATENDIDA_SOLVER_TOTAL", "QTD_NAO_ATEND_SOLVER_TOTAL", "DIF_FECHAMENTO_DEMANDA",
        "QTD_ITENS_DEMANDA", "QTD_ITENS_COM_PRODUCAO", "QTD_ITENS_SEM_PRODUCAO", "QTD_ITENS_COM_NAO_ATENDIMENTO",
        "QTD_ALOC_REC", "QTD_ALOC_REC_ESTOURO", "ESTOURO_HR_TOTAL", "HOR_CAP_TOTAL", "HR_PRODUZIR_SOLVER_TOTAL", "OCUPACAO_GLOBAL",
        "QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS_PRODUCAO", "QTD_ALTERNATIVAS_HOR_CAP_ZERO", "QTD_LINHAS_RASTRO_RATEIO", "QTD_ITENS_DIAGNOSTICO"
    ],
    "VALOR": [
        nec_total, atend_total, nao_atend_total, dif_fechamento,
        len(nao_atend_item), (nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL).sum(), (nao_atend_item["QTD_ATENDIDA_SOLVER"] <= TOL).sum(), (nao_atend_item["QTD_NAO_ATEND_SOLVER"] > TOL).sum(),
        len(resumo_maquinas), (resumo_maquinas["ESTOURO_HR_SOLVER"] > TOL).sum(), resumo_maquinas["ESTOURO_HR_SOLVER"].sum(), resumo_maquinas["HOR_CAP"].sum(), resumo_maquinas["HR_PRODUZIR_SOLVER"].sum(), resumo_maquinas["HR_PRODUZIR_SOLVER"].sum() / resumo_maquinas["HOR_CAP"].sum() if resumo_maquinas["HOR_CAP"].sum() > 0 else 0,
        len(bd_alternativas_item), len(bd_alternativas_validas), (bd_alternativas_item["HOR_CAP"] <= 0).sum(), len(rastro_rateio), len(diagnostico_item)
    ]
})

display(validacao)


In [ ]:

# ============================================================
# 07. EXPORTAÇÃO
# ============================================================

OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    resumo_maquinas.to_excel(writer, sheet_name="01_RESUMO_MAQUINAS", index=False)
    plano_producao.to_excel(writer, sheet_name="02_PLANO_PRODUCAO", index=False)
    nao_atend_item.to_excel(writer, sheet_name="03_NAO_ATEND_ITEM", index=False)
    auditoria_capacidade.to_excel(writer, sheet_name="04_AUDITORIA_CAPACIDADE", index=False)
    alternativas_item.to_excel(writer, sheet_name="05_ALTERNATIVAS_ITEM", index=False)
    rastro_rateio.to_excel(writer, sheet_name="06_RASTRO_RATEIO", index=False)
    diagnostico_item.to_excel(writer, sheet_name="07_DIAGNOSTICO_ITEM", index=False)
    diagnostico_alternativas.to_excel(writer, sheet_name="08_DIAGNOSTICO_ALTERNATIVAS", index=False)
    validacao.to_excel(writer, sheet_name="99_VALIDACAO", index=False)

print("Arquivo exportado:", OUTPUT_SOLVER_FILE)

wb = load_workbook(OUTPUT_SOLVER_FILE, read_only=True)
print("Guias exportadas:")
for s in wb.sheetnames:
    print("-", s)


In [ ]:

# ============================================================
# 08. CHECK RÁPIDO DE UM PRODUTO ESPECÍFICO (OPCIONAL)
# ============================================================

COD_PROD_CHECK = "0103"
mask_cod = lambda df: df["COD_PROD"].astype(str).str.zfill(4).eq(COD_PROD_CHECK.zfill(4))

print("Plano de produção do produto:", COD_PROD_CHECK)
display(plano_producao[mask_cod(plano_producao)].sort_values(["ID_PROD_UNID_FAT", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]))

print("Não atendimento do produto:", COD_PROD_CHECK)
display(nao_atend_item[mask_cod(nao_atend_item)].sort_values(["ID_PROD_UNID_FAT"]))

print("Diagnóstico do produto:", COD_PROD_CHECK)
display(diagnostico_item[mask_cod(diagnostico_item)].sort_values(["ID_PROD_UNID_FAT"]))

print("Rastro do produto:", COD_PROD_CHECK)
display(rastro_rateio[mask_cod(rastro_rateio)].sort_values(["ID_PROD_UNID_FAT", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]))

print("Alternativas do produto:", COD_PROD_CHECK)
display(diagnostico_alternativas[mask_cod(diagnostico_alternativas)].sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]))
